# 3 pamoka - Agentiniai dizaino šablonai

Šioje pamokoje nagrinėsime tris pagrindinius dizaino šablonus, skirtus veiksmingų dirbtinio intelekto agentų kūrimui:

1. **Aiškios agento instrukcijos** — tiksliai apibrėžti vaidmenį nurodantys užklausimai, kurie valdo agento elgesį
2. **Struktūruotas išvestis su Pydantic modeliais** — užtikrinantys, kad agentai grąžintų numatomus, patikrintus duomenis
3. **Vienos atsakomybės agentai** — projektuojantys agentus, sukurtus konkrečiam darbui atlikti gerai

Taikysime kiekvieną šabloną **kelionių tikslų rekomendacijų** scenarijui, palaipsniui kuriant sistemą, galinčią siūlyti tikslus, tikrinti prieinamumą ir tvarkyti logistiką.


## Nustatymas


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Šablonas 1: Aiškios agento instrukcijos

Įtakingiausias šablonas taip pat yra pats paprasčiausias: rašyti aiškias, detalias instrukcijas savo agentui.

Geros instrukcijos apibrėžia:
- **Kas** yra agentas (asmens bruožas ir tonas)
- **Ką** jis turėtų daryti (žingsnis po žingsnio atsakomybės)
- **Kaip** jis turėtų elgtis (apribojimai ir stilius)

Žemiau sukuriame kelionių konsjeržo agentą su aiškiomis instrukcijomis, kurios formuoja kiekvieną jo atsakymą.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Šablonas 2: Struktūruotas išėjimas su Pydantic modeliais

Laisvos formos tekstas naudingas pokalbiams, tačiau tolesnės sistemos turi gauti struktūrizuotus duomenis.
Naudodami **Pydantic modelius** kartu su **įrankio funkcija**, galime:

- Apibrėžti tikslų agento išėjimo schemą
- Automatiškai tikrinti atsakymus
- Patikimai integruoti agento rezultatus į programos logiką

Svarbiausia taikant apribojimus yra perduoti `response_format` paleidžiant agentą. Tai priverčia
modelį grąžinti patikrintą `TravelRecommendations` objektą (pasiekiamą per `response.value`)
vietoje laisvos formos teksto. Įrankis `get_destination_details` taip pat grąžina tipišką
`DestinationRecommendation`, todėl duomenys lieka struktūruoti nuo pradžios iki pabaigos.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Šablonas 3: Vienos Atsakomybės Agentai

Sudėtingi uždaviniai naudingai sprendžiami paskirstant darbą keliems specializuotiems agentams, kiekvienas turintis vieną atsakomybę:

- **Kelionės tikslų ekspertas**, kuris žino apie vietas ir jų prieinamumą
- **Logistikos planuotojas**, kuris rūpinasi skrydžiais, viešbučiais ir maršrutais

Tai atitinka programinės įrangos inžinerijos principą *atsakomybių atskyrimas* — kiekvieną agentą lengviau testuoti, prižiūrėti ir tobulinti nepriklausomai.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Santrauka

Šioje pamokoje taikėme tris agentūros dizaino šablonus kelionių rekomendacijų scenarijui:

| Šablonas | Pagrindinė idėja | Privalumas |
|---|---|---|
| **Aiškios instrukcijos** | Iš anksto apibrėžti asmenybę, atsakomybes ir apribojimus | Nuoseklus, prekės ženklui atitinkantis agento elgesys |
| **Struktūruotas rezultatas** | Naudoti Pydantic modelius kaip atsakymo formatą | Patvirtinti, mašinai skaitomi rezultatai |
| **Vienintelė atsakomybė** | Suteikti kiekvienam agentui vieną aiškią užduotį | Lengviau testuoti, prižiūrėti ir komponuoti |

Šie šablonai natūraliai derinami — galite sujungti aiškias instrukcijas su struktūruotu rezultatu vienos atsakomybės agente, kad sukurtumėte tvirtas, gamybai paruoštas sistemas.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
